#### **1. Setup**


In [11]:
from pyspark.sql.functions import broadcast, col, when, lit, current_timestamp, expr
from delta.tables import DeltaTable


StatementMeta(, 6b3ff08a-76b0-4c43-a56a-aeeafd5230c6, 13, Finished, Available, Finished, False)

In [12]:
# Read Data from Lakehouse
df_new_orders = spark.table("LH_Nilay.bronze.bronze_orders")
df_customer = spark.table("LH_Nilay.bronze.bronze_customer")
df_rules = spark.table("LH_Nilay.meta.dq_rules")

# Enterprise Logic: Combine new incoming orders with existing Pending & Quarantine queues
df_orders = df_new_orders

# 1. Add Pending Queue (Late Arriving Data)
try:
    # Force direct read from Delta log to avoid Spark caching issues
    df_pending = spark.read.format("delta").table("LH_Nilay.audit.pending_queue").select(*df_new_orders.columns)
    df_orders = df_orders.unionByName(df_pending, allowMissingColumns=True)
except Exception as e:
    pass

# 2. Add Quarantine Queue (Auto-Recovery for Rule Changes)
try:
    # Force direct read from Delta log to avoid Spark caching issues
    df_quarantine = spark.read.format("delta").table("LH_Nilay.audit.quarantine_orders").select(*df_new_orders.columns)
    df_orders = df_orders.unionByName(df_quarantine, allowMissingColumns=True)
except Exception as e:
    pass

# THE ENTERPRISE FIX: Drop duplicates by OrderID to prevent double-processing during retries
df_orders = df_orders.dropDuplicates(["OrderID"])

StatementMeta(, 6b3ff08a-76b0-4c43-a56a-aeeafd5230c6, 14, Finished, Available, Finished, False)

#### **2. Smart Routing & DQ Firewall Engine**

In [13]:
# 1. Broadcast Join to check missing customer reference
df_joined = df_orders.join(broadcast(df_customer.drop("LastModifiedDate")), "CustomerID", "left")

# 2. Dynamically fetch and parse rules from the meta table
rules = df_rules.collect()

quarantine_cases = [] 
warning_cases = [] 
warn_exprs = [] 
pending_dynamic_reason = "Missing_Reference" 

for row in rules: 
    if str(row["ActiveFlag"]).lower() != "true": 
        continue 
        
    column = row["ColumnName"] 
    rule_type = row["RuleType"] 
    threshold = row["ThresholdValue"] 
    action = row["Action"] 
    reason = row["RejectionReason"] 
    
    if rule_type == "ExistsInRef": 
        pending_dynamic_reason = reason 
        continue 
        
    condition = "" 
    if rule_type == "GreaterThan": 
        condition = f"NOT ({column} > {threshold})" 
    elif rule_type == "LessThanOrEqual": 
        condition = f"NOT ({column} <= {threshold})" 
    elif rule_type == "IsNotNull": 
        condition = f"{column} IS NULL" 
        
    if condition and action == "QUARANTINE": 
        quarantine_cases.append(f"CASE WHEN {condition} THEN '{reason}' END") 
    elif condition and action == "WARN": 
        warning_cases.append(f"CASE WHEN {condition} THEN '{reason}' END") 
        warn_exprs.append(condition) 

# 3. Combine dynamic conditions
final_warn = " OR ".join(warn_exprs) if warn_exprs else "1=0" 
rejection_expr = expr(f"NULLIF(concat_ws(', ', {', '.join(quarantine_cases)}), '')") if quarantine_cases else lit(None).cast("string")
warning_expr = expr(f"NULLIF(concat_ws(', ', {', '.join(warning_cases)}), '')") if warning_cases else lit(None).cast("string")

# 4. Apply dynamic routing & Pipeline Run Time
df_evaluated = df_joined.withColumn(
    "Rejection_Reason",
    when(col("CustomerName").isNull() & col("CustomerID").isNotNull(), lit(pending_dynamic_reason))
    .otherwise(rejection_expr)
).withColumn(
    "DQ_Status",
    when(col("Rejection_Reason") == pending_dynamic_reason, lit("PENDING"))
    .when(col("Rejection_Reason").isNotNull(), lit("QUARANTINE"))
    .otherwise(lit("SILVER"))
).withColumn(
    "Warn_Flag",
    when(expr(final_warn), lit(True)).otherwise(lit(False))
).withColumn(
    "Warning_Reason",
    warning_expr
).withColumn("Batch_Run_Time", current_timestamp())

# THE ENTERPRISE FIX (SCALABLE): Materialize to Staging Table 
# This eliminates Lazy Evaluation bugs & scales to 100M+ rows with zero RAM bottlenecks

staging_table = "LH_Nilay.audit.stg_evaluated_orders"
df_evaluated.write.format("delta").mode("overwrite").saveAsTable(staging_table)

# Read the materialized data back for all downstream operations
df_materialized = spark.read.format("delta").table(staging_table)

StatementMeta(, 6b3ff08a-76b0-4c43-a56a-aeeafd5230c6, 15, Finished, Available, Finished, False)

#### **3. Destination Routing & Write to Lakehouse**

In [14]:
# 1. Split Data based on Status from the Materialized Table
df_silver = df_materialized.filter(col("DQ_Status") == "SILVER").drop("Rejection_Reason", "DQ_Status") 
df_quarantine = df_materialized.filter(col("DQ_Status") == "QUARANTINE").drop("DQ_Status") 
df_pending = df_materialized.filter(col("DQ_Status") == "PENDING").drop("DQ_Status") 

def write_or_merge_delta(df, table_name): 
    if spark.catalog.tableExists(table_name): 
        DeltaTable.forName(spark, table_name).alias("target").merge(
            df.alias("source"),
            "target.OrderID = source.OrderID"
        ).whenNotMatchedInsertAll().execute()
    else: 
        df.write.format("delta").mode("overwrite").saveAsTable(table_name) 

# 2. Execute Destination Routing
write_or_merge_delta(df_silver, "LH_Nilay.silver.silver_orders") 
write_or_merge_delta(df_quarantine, "LH_Nilay.audit.quarantine_orders") 
write_or_merge_delta(df_pending, "LH_Nilay.audit.pending_queue") 

# 3. Add Batch_Run_Time to Customer Dimension and Write
df_customer_final = df_customer.withColumn("Batch_Run_Time", current_timestamp()) 
if spark.catalog.tableExists("LH_Nilay.silver.silver_customer"): 
    DeltaTable.forName(spark, "LH_Nilay.silver.silver_customer").alias("t").merge(
        df_customer_final.alias("s"), 
        "t.CustomerID = s.CustomerID"
    ).whenNotMatchedInsertAll().execute()
else: 
    df_customer_final.write.format("delta").mode("overwrite").saveAsTable("LH_Nilay.silver.silver_customer") 

# 4. THE MAGIC CLEANUP for Quarantine & Pending Tables
tables_to_cleanup = ["LH_Nilay.audit.quarantine_orders", "LH_Nilay.audit.pending_queue"]

for table_name in tables_to_cleanup:
    if spark.catalog.tableExists(table_name): 
        DeltaTable.forName(spark, table_name).alias("t").merge(
            df_silver.alias("s"),
            "t.OrderID = s.OrderID"
        ).whenMatchedDelete().execute()

StatementMeta(, 6b3ff08a-76b0-4c43-a56a-aeeafd5230c6, 16, Finished, Available, Finished, False)

#### **4. Audit Log Generation**

In [15]:
# 1. Fetch dynamic threshold from config table
config_df = spark.table("LH_Nilay.meta.pipeline_config").filter(col("PipelineName") == "PL_Master_DQ_Execution").collect() 
threshold_limit = config_df[0]["ErrorThresholdLimit"] if config_df else 0.10 

# 2. Calculate execution metrics (Using Materialized Dataframes to guarantee accurate counts)
total_records = df_materialized.count() 
passed_records = df_silver.count() 
failed_records = df_quarantine.count() 
pending_records = df_pending.count() 
warning_records = df_silver.filter(col("Warn_Flag") == True).count() 

# 3. Dynamic Threshold Check
error_rate = round(failed_records / total_records, 2) if total_records > 0 else 0 

if error_rate == 0: 
    batch_status = "PERFECT_CLEAN" 
elif error_rate > threshold_limit: 
    batch_status = "THRESHOLD_BREACHED" 
else: 
    batch_status = "WITHIN_LIMIT"

# 4. Create and Append Audit Log
audit_data = [(
    "PL_Master_DQ_Execution", total_records, passed_records, warning_records, 
    failed_records, pending_records, batch_status, float(error_rate), float(threshold_limit)
)] 

df_audit_log = spark.createDataFrame(
    audit_data, 
    ["PipelineName", "TotalProcessed", "Passed", "Warnings", "Failed", "Pending", "BatchStatus", "ErrorRate", "ConfiguredThreshold"]
).withColumn("ExecutionTime", current_timestamp()) 

# 5. Append to Audit Log Table
df_audit_log.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("LH_Nilay.audit.dq_execution_log")

StatementMeta(, 6b3ff08a-76b0-4c43-a56a-aeeafd5230c6, 17, Finished, Available, Finished, False)